# 03 — Visualise Results

This notebook covers:
1. Probability map visualisation
2. Binary mask at multiple thresholds
3. Change detection map (gain / loss / stable)
4. Kalman-smoothed area time series
5. Validation metric curves (ROC, PR)


In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import rasterio
from pathlib import Path

from scripts.utils import load_config
cfg = load_config('../config.yaml')
print('Config OK.')

In [ ]:
# ── Probability map ──────────────────────────────────────────────────────────
def percentile_stretch(arr, lo=2, hi=98):
    p_lo, p_hi = np.percentile(arr, [lo, hi])
    return np.clip((arr - p_lo) / (p_hi - p_lo + 1e-6), 0, 1)

prob_dir = Path('../data/outputs/probability_maps')
prob_files = sorted(prob_dir.glob('*_prob.tif'))

if prob_files:
    with rasterio.open(prob_files[-1]) as src:
        prob_map = src.read(1).astype(np.float32)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    im = axes[0].imshow(prob_map, cmap='RdYlBu_r', vmin=0, vmax=1)
    plt.colorbar(im, ax=axes[0], fraction=0.046, label='Gully Probability')
    axes[0].set_title(f'Probability Map — {prob_files[-1].name}')
    axes[0].axis('off')
    
    axes[1].hist(prob_map.ravel(), bins=100, color='#e74c3c', alpha=0.75)
    axes[1].set_xlabel('Probability')
    axes[1].set_ylabel('Pixel count')
    axes[1].set_title('Probability Distribution')
    
    plt.tight_layout()
    plt.savefig('../data/outputs/reports/probability_map_preview.png', dpi=120, bbox_inches='tight')
    plt.show()
else:
    print('No probability maps. Run: python main.py --step infer')

In [ ]:
# ── Binary masks at multiple thresholds ──────────────────────────────────────
if prob_files:
    thresholds = [0.3, 0.5, 0.7]
    fig, axes = plt.subplots(1, len(thresholds) + 1, figsize=(18, 4))
    
    axes[0].imshow(prob_map, cmap='RdYlBu_r', vmin=0, vmax=1)
    axes[0].set_title('Probability')
    axes[0].axis('off')
    
    for i, thr in enumerate(thresholds):
        binary = (prob_map >= thr).astype(np.uint8)
        n_pos = binary.sum()
        axes[i+1].imshow(binary, cmap='binary')
        axes[i+1].set_title(f'thr={thr}  ({n_pos:,} px)')
        axes[i+1].axis('off')
    
    plt.suptitle('Binary Masks at Different Thresholds', fontsize=13)
    plt.tight_layout()
    plt.savefig('../data/outputs/reports/binary_mask_comparison.png', dpi=120, bbox_inches='tight')
    plt.show()

In [ ]:
# ── Change detection visualisation ───────────────────────────────────────────
change_dir = Path('../data/outputs/change_detection')
combined_files = sorted(change_dir.glob('*_combined.tif'))

if combined_files:
    with rasterio.open(combined_files[-1]) as src:
        combined = src.read().astype(np.float32)   # (3, H, W): gain/loss/stable
    
    # Build a colour composite: gain=green, loss=red, stable=yellow
    H, W = combined.shape[1], combined.shape[2]
    rgb = np.zeros((H, W, 3), dtype=np.float32)
    rgb[:, :, 1] = combined[0]   # gain → green
    rgb[:, :, 0] = combined[1]   # loss → red
    rgb[:, :, 0] += combined[2] * 0.8  # stable → orange-yellow
    rgb[:, :, 1] += combined[2] * 0.6
    rgb = np.clip(rgb, 0, 1)
    
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    labels = ['Gain (new gullies)', 'Loss (healed)', 'Stable', 'Combined']
    cmaps = ['Greens', 'Reds', 'Oranges', None]
    
    for i, (label, cmap) in enumerate(zip(labels, cmaps)):
        if i < 3:
            axes[i].imshow(combined[i], cmap=cmap, vmin=0, vmax=1)
        else:
            axes[i].imshow(rgb)
        axes[i].set_title(label)
        axes[i].axis('off')
    
    plt.suptitle(f'Change Detection — {combined_files[-1].name}', fontsize=13)
    plt.tight_layout()
    plt.savefig('../data/outputs/reports/change_detection_preview.png', dpi=120, bbox_inches='tight')
    plt.show()
else:
    print('No change maps. Run: python main.py --step change')

In [ ]:
# ── Kalman-smoothed area time series ─────────────────────────────────────────
kalman_path = Path('../data/outputs/change_detection/kalman_smoothed.json')
if kalman_path.exists():
    with open(kalman_path) as f:
        kal = json.load(f)
    
    times = kal['times']
    raw = kal['raw_areas']
    smoothed = kal['smoothed_areas']
    velocities = kal.get('velocities', [])
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    axes[0].scatter(times, raw, color='#e74c3c', zorder=5, label='Observed')
    axes[0].plot(times, smoothed, color='#2980b9', linewidth=2, label='Kalman Smoothed')
    axes[0].fill_between(times,
                          [s - 0.1 for s in smoothed],
                          [s + 0.1 for s in smoothed],
                          alpha=0.2, color='#2980b9', label='±1σ')
    axes[0].set_xlabel('Year')
    axes[0].set_ylabel('Gully Area (ha)')
    axes[0].set_title('Gully Area Time Series')
    axes[0].legend()
    
    if velocities:
        colors = ['#27ae60' if v >= 0 else '#e74c3c' for v in velocities]
        axes[1].bar(times, velocities, color=colors, alpha=0.75)
        axes[1].axhline(0, color='grey', linestyle='--')
        axes[1].set_xlabel('Year')
        axes[1].set_ylabel('Velocity (ha/yr)')
        axes[1].set_title('Expansion Velocity')
    
    plt.tight_layout()
    plt.savefig('../data/outputs/reports/time_series.png', dpi=120, bbox_inches='tight')
    plt.show()
else:
    print('No Kalman data. Run: python main.py --step change')

In [ ]:
# ── Validation ROC and PR curves ─────────────────────────────────────────────
spatial_cv_path = Path('../data/outputs/reports/spatial_cv.json')
if spatial_cv_path.exists():
    with open(spatial_cv_path) as f:
        cv = json.load(f)
    
    agg = cv.get('aggregate', {})
    print('=== Spatial Block CV Results ===')
    for k in ['iou', 'f1', 'precision', 'recall', 'accuracy', 'roc_auc', 'ap']:
        if k in agg:
            a = agg[k]
            print(f'  {k:<12}: {a["mean"]:.4f} ± {a["std"]:.4f}  (range [{a["min"]:.4f}, {a["max"]:.4f}])')
    
    # Target thresholds from paper
    print('\nTarget metrics:')
    targets = {'iou': 0.65, 'f1': 0.75, 'precision': 0.70, 'recall': 0.72}
    for k, target in targets.items():
        if k in agg:
            achieved = agg[k]['mean']
            status = '✓' if achieved >= target else '✗'
            print(f'  {k:<12}: {achieved:.4f} vs target {target:.2f}  {status}')
else:
    print('No CV results. Run: python main.py --step validate')